In [3]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

## UPL Goal Data

In [56]:
headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0'}

In [57]:
g_url = 'https://upl.co.ug/calendar/2025-26-fixtures-results/'
g_response = requests.get(g_url,headers=headers)
g_response.status_code

200

In [58]:
g_soup = BeautifulSoup(g_response.content, 'html.parser')

In [59]:
matches_table = g_soup.select_one('div[class="sp-template sp-template-event-blocks"]')

In [60]:
match_urls = []  # Create a list to store URLs

for i in matches_table.find_all('a'):
    match_url = i.get('href')
    if match_url and match_url.startswith('https://upl.co.ug/event/'):
        match_urls.append(match_url)

match_urls = list(set(match_urls))  # Remove duplicates

all_rows = []  # accumulate rows across all matches

for url in match_urls:
    try:
        ind_response = requests.get(url, headers=headers, timeout=10)
    except Exception:
        # skip this URL if request fails
        continue
    soup = BeautifulSoup(ind_response.content, 'html.parser')

    timeline = soup.select_one('div[class="sp-template sp-template-timeline sp-template-event-timeline sp-template-vertical-timeline"]')
    if not timeline:
        continue  # Skip if no timeline found

    goals = []
    for icon in timeline.find_all('i', class_='sp-icon-soccerball'):
        tr = icon.find_parent('tr')
        if not tr:
            continue
        tds = tr.find_all('td')
        minute = tds[1].get_text(strip=True) if len(tds) > 1 else ''
        # normalize goal type detection (check for explicit markers)
        icon_title = (icon.get('title') or '').strip().lower()
        if icon_title == 'own goals' or 'own goal' in icon_title:
            goal_type = 'Own Goal'
        elif '(P)' in minute or minute.strip().upper().endswith('P'):
            goal_type = 'Penalty'
        elif 'OG' in minute.upper():
            goal_type = 'Own Goal'
        else:
            goal_type = 'Regular'
        minute = minute.replace("'", "").replace("(P)", "").replace("OG", "").strip()

        # determine which side cell contains the icon (home=tds[0], away=tds[2])
        icon_td = icon.find_parent('td')
        side_cell_index = tds.index(icon_td) if (icon_td and icon_td in tds) else 2
        side = 'home' if side_cell_index == 0 else 'away'

        player_text = tds[side_cell_index].get_text(" ", strip=True)
        # remove jersey numbers like "11." or "11. "
        player_text = re.sub(r'^\s*\d+\.*\s*', '', player_text).strip()
        goals.append({'goal_minute': minute, 'team_side': side, 'player_name': player_text, 'goal_type': goal_type})

    # get teams safely
    title_tag = soup.select_one('h1[class="entry-title"]')
    if title_tag:
        teams = title_tag.get_text(strip=True).split(' vs ')
        home_team = teams[0] if len(teams) > 0 else None
        away_team = teams[1] if len(teams) > 1 else None
    else:
        home_team = away_team = None

    event_details_table = soup.select_one('table[class="sp-event-details sp-data-table sp-scrollable-table"]')

    # Safely get headers/data for the event details table (do NOT overwrite the
    # global HTTP request 'headers' dict)
    match_details = {}
    if event_details_table:
        event_table_headers = [th.get_text(strip=True) for th in event_details_table.select('th')]
        # Get data from tbody (flattened list of td values)
        data = [td.get_text(strip=True) for td in event_details_table.select('tbody td')]
        # Create dictionary by pairing event_table_headers with data
        match_details = dict(zip(event_table_headers, data))

    # attach teams (separately from the event details)
    match_details.update({'home_team': home_team, 'away_team': away_team})

    # If there are goals, create one row per goal; otherwise add one row with empty goal fields
    if goals:
        for goal in goals:
            row = {**match_details, **goal}  # Merge match details with goal details
            # ensure row is a dict (defensive)
            if isinstance(row, dict):
                all_rows.append(row)
    else:
        # Add at least one row so match appears even when there are no goals
        empty_goal_row = {'goal_minute': None, 'team_side': None, 'player_name': None, 'goal_type': None}
        row = {**match_details, **empty_goal_row}
        all_rows.append(row)

# After processing all matches, create a single DataFrame
if all_rows:
    df = pd.DataFrame(all_rows)
    # Reorder columns to have match details first followed by goal details
    goal_cols = ['goal_minute', 'team_side', 'player_name', 'goal_type']
    match_cols = [col for col in df.columns if col not in goal_cols]
    # keep only existing goal columns (in case some are missing)
    existing_goal_cols = [c for c in goal_cols if c in df.columns]
    df = df[match_cols + existing_goal_cols]
else:
    # empty dataframe with expected columns
    df = pd.DataFrame(columns=['home_team', 'away_team', 'goal_minute', 'team_side', 'player_name', 'goal_type'])


In [61]:
df

,Date,Time,League,Season,Match Day,home_team,away_team,goal_minute,team_side,player_name,goal_type
0,29/10/2025,4:00 pm,Uganda Premier League,2025/26,5,Police FC,Maroons FC,18,home,Bedia Djuma Ikamba,Regular
1,29/10/2025,4:00 pm,Uganda Premier League,2025/26,5,Police FC,Maroons FC,28,home,Bedia Djuma Ikamba,Regular
2,29/10/2025,4:00 pm,Uganda Premier League,2025/26,5,Police FC,Maroons FC,42,away,Rogers Kiwanuka,Regular
3,01/11/2025,4:00 pm,Uganda Premier League,2025/26,5,Mbarara City FC,NEC FC,54,away,Allan Mugalu,Regular
4,05/11/2025,8:00 pm,Uganda Premier League,2025/26,6,Express FC,Mbarara City FC,12,home,Abdulahi Rashid Kawawa,Regular
...,...,...,...,...,...,...,...,...,...,...,...
92,31/10/2025,4:00 pm,Uganda Premier League,2025/26,5,Buhimba United Saints FC,SC Villa,2,away,Reagan Mpande,Regular
93,31/10/2025,4:00 pm,Uganda Premier League,2025/26,5,Buhimba United Saints FC,SC Villa,14,away,Aslam Ssemakula,Regular
94,31/10/2025,4:00 pm,Uganda Premier League,2025/26,5,Buhimba United Saints FC,SC Villa,65,away,Reagan Mpande,Regular
95,01/10/2025,4:00 pm,Uganda Premier League,2025/26,2,BUL FC,Maroons FC,37,home,Ibrahim Mugulusi,Regular


In [62]:
df.to_csv('upl_goals_2025_26.csv', index=False)